# 🤖 KMUT Models — Load & Use Guide
## Kalaignar Magalir Urimai Thittam | Trained Model Bundle

This notebook shows how to **load and use every model** from `kmut_models.pkl`.

| # | Model Key | Type | Task |
|---|-----------|------|------|
| 1 | `rf_eligibility` | Random Forest | Eligibility prediction |
| 2 | `gb_eligibility` | Gradient Boosting | Eligibility prediction |
| 3 | `dt_eligibility` | Decision Tree | Eligibility prediction |
| 4 | `lr_eligibility` | Logistic Regression | Eligibility prediction |
| 5 | `gb_approval` | Gradient Boosting | Approval prediction |
| 6 | `rf_approval` | Random Forest | Approval prediction |
| 7 | `kmeans` | K-Means (K=5) | Beneficiary segmentation |
| 8 | `pca` | PCA 2D | Cluster visualization |
| 9 | `isolation_forest` | Isolation Forest | Anomaly/fraud detection |
| 10 | `tfidf_vectorizer` | TF-IDF | RAG policy Q&A |

> **Requires:** `kmut_models.pkl` in same folder. Install: `pip install scikit-learn pandas numpy`

## 📦 1. Load the Model Bundle

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load all models at once
with open('kmut_models.pkl', 'rb') as f:
    models = pickle.load(f)

print("Models loaded successfully!")
print(f"Available keys ({len(models)}):")
for key in models:
    print(f"  • {key}")

## 🔍 2. Inspect Model Metadata

In [ ]:
# View model info for every trained model
print(f"{'Model Key':<22} {'Type':<30} {'Task'}")
print("-" * 90)
for key, info in models['model_info'].items():
    print(f"{key:<22} {info['type']:<30} {info['task']}")

## ✅ 3. Eligibility Prediction
### 3a. Random Forest (Best — AUC 1.0)

In [ ]:
# Extract model and feature list
rf_elig      = models['rf_eligibility']
features     = models['features_eligibility']
le_dict      = models['label_encoders']

print("Eligibility feature list:")
for i, f in enumerate(features, 1):
    print(f"  {i:2}. {f}")

In [ ]:
# ── Helper: prepare a single applicant record ────────────────────────────
def prepare_eligibility_input(record: dict) -> np.ndarray:
    """
    record: dict with raw applicant fields.
    Returns numpy array ready for eligibility models.
    """
    # Derived fields
    record['total_land_acres']      = record.get('wet_land_acres', 0) + record.get('dry_land_acres', 0)
    record['income_threshold_ratio']= record['annual_income_inr'] / 250000
    record['ineligibility_count']   = sum([
        record.get('has_four_wheeler', False),
        record.get('has_govt_employee_family', False),
        record.get('has_income_tax_filer', False),
        record.get('has_gst_business', False),
        record.get('already_receiving_pension', False),
    ])
    record['vulnerability_index'] = (
        int(record['annual_income_inr'] < 100000) * 3 +
        int(record.get('is_bpl', False)) * 2 +
        int(record.get('marital_status', '') in ['Widowed', 'Divorced/Separated']) * 2 +
        int(not record.get('has_bank_account', True)) +
        int(record.get('digital_literacy', '') == 'None') +
        int(not record.get('has_smartphone', True)) +
        int(record.get('education_level', '') == 'Illiterate') * 2
    )
    # Encode categoricals
    cat_cols = ['district','area_type','marital_status','social_category',
                'education_level','occupation','bank_name','digital_literacy',
                'hof_gender']
    for col in cat_cols:
        enc_col = f'{col}_enc'
        le = le_dict.get(col)
        val = record.get(col, 'unknown')
        try:
            record[enc_col] = int(le.transform([val])[0])
        except:
            record[enc_col] = 0  # unseen label fallback

    row = [record.get(f, 0) for f in features]
    return np.array(row).reshape(1, -1)

# ── Sample applicant ─────────────────────────────────────────────────────
sample_applicant = {
    'age': 38,
    'family_size': 4,
    'annual_income_inr': 85000,
    'wet_land_acres': 1.2,
    'dry_land_acres': 2.0,
    'electricity_units_year': 2100,
    'has_four_wheeler': False,
    'has_govt_employee_family': False,
    'has_income_tax_filer': False,
    'has_gst_business': False,
    'already_receiving_pension': False,
    'district': 'Madurai',
    'area_type': 'Rural',
    'marital_status': 'Widowed',
    'social_category': 'MBC',
    'education_level': 'Secondary (9-10)',
    'occupation': 'Agricultural Laborer',
    'bank_name': 'Indian Bank',
    'digital_literacy': 'Basic',
    'hof_gender': 'Female',
    'is_bpl': True,
    'has_bank_account': True,
    'has_smartphone': False,
}

X_input = prepare_eligibility_input(sample_applicant.copy())

# Predict with Random Forest
pred  = rf_elig.predict(X_input)[0]
proba = rf_elig.predict_proba(X_input)[0]

print("=" * 50)
print("ELIGIBILITY PREDICTION (Random Forest)")
print("=" * 50)
print(f"Eligible:         {'✅ YES' if pred == 1 else '❌ NO'}")
print(f"Confidence:       {max(proba)*100:.1f}%")
print(f"P(Eligible):      {proba[1]*100:.1f}%")
print(f"P(Not Eligible):  {proba[0]*100:.1f}%")

### 3b. Compare All 4 Eligibility Models

In [ ]:
X_input = prepare_eligibility_input(sample_applicant.copy())
scaler_e = models['scaler_eligibility']
X_scaled = scaler_e.transform(X_input)

elig_models = {
    'Random Forest':     (models['rf_eligibility'],  X_input),
    'Gradient Boosting': (models['gb_eligibility'],  X_input),
    'Decision Tree':     (models['dt_eligibility'],  X_input),
    'Logistic Regress.': (models['lr_eligibility'],  X_scaled),
}

print(f"{'Model':<22} {'Prediction':<14} {'P(Eligible)'}")
print("-" * 50)
for name, (model, X) in elig_models.items():
    pred  = model.predict(X)[0]
    proba = model.predict_proba(X)[0][1]
    label = '✅ Eligible' if pred == 1 else '❌ Not Eligible'
    print(f"{name:<22} {label:<14} {proba*100:.1f}%")

## 📋 4. Approval Prediction
*(Among applicants who have already applied)*

In [ ]:
def prepare_approval_input(record: dict) -> np.ndarray:
    """Prepare input for approval prediction models."""
    features_appr = models['features_approval']
    le_dict       = models['label_encoders']

    record['total_land_acres']       = record.get('wet_land_acres',0) + record.get('dry_land_acres',0)
    record['has_land']               = record['total_land_acres'] > 0
    record['income_threshold_ratio'] = record['annual_income_inr'] / 250000
    record['ineligibility_count']    = sum([
        record.get('has_four_wheeler', False), record.get('has_govt_employee_family', False),
        record.get('has_income_tax_filer', False), record.get('has_gst_business', False),
        record.get('already_receiving_pension', False),
    ])
    record['vulnerability_index'] = (
        int(record['annual_income_inr'] < 100000)*3 + int(record.get('is_bpl',False))*2 +
        int(record.get('marital_status','') in ['Widowed','Divorced/Separated'])*2 +
        int(not record.get('has_bank_account',True)) + int(record.get('digital_literacy','')=='None') +
        int(not record.get('has_smartphone',True)) + int(record.get('education_level','')=='Illiterate')*2
    )
    record['digital_inclusion_score'] = (
        int(record.get('has_bank_account',False))*2 + int(record.get('has_smartphone',False))*2
    )
    record['is_direct_hof_female'] = record.get('hof_gender','') == 'Female'

    cat_cols = ['district','area_type','marital_status','social_category',
                'education_level','occupation','bank_name','digital_literacy','hof_gender']
    for col in cat_cols:
        le = le_dict.get(col)
        val = record.get(col,'unknown')
        try:    record[f'{col}_enc'] = int(le.transform([val])[0])
        except: record[f'{col}_enc'] = 0

    return np.array([record.get(f,0) for f in features_appr]).reshape(1,-1)


X_appr = prepare_approval_input(sample_applicant.copy())

for name, key in [('Gradient Boosting', 'gb_approval'), ('Random Forest', 'rf_approval')]:
    model = models[key]
    pred  = model.predict(X_appr)[0]
    proba = model.predict_proba(X_appr)[0]
    print(f"{name} Approval Prediction:")
    print(f"  Decision:       {'✅ APPROVED' if pred==1 else '❌ NOT APPROVED'}")
    print(f"  P(Approved):    {proba[1]*100:.1f}%")
    print()

## 🗂️ 5. Beneficiary Segmentation (K-Means + PCA)

In [ ]:
def prepare_cluster_input(record: dict) -> np.ndarray:
    le_dict = models['label_encoders']
    record['total_land_acres'] = record.get('wet_land_acres',0) + record.get('dry_land_acres',0)
    record['vulnerability_index'] = (
        int(record['annual_income_inr']<100000)*3 + int(record.get('is_bpl',False))*2 +
        int(record.get('marital_status','') in ['Widowed','Divorced/Separated'])*2 +
        int(not record.get('has_bank_account',True)) + int(record.get('digital_literacy','')=='None') +
        int(not record.get('has_smartphone',True)) + int(record.get('education_level','')=='Illiterate')*2
    )
    record['digital_inclusion_score'] = (
        int(record.get('has_bank_account',False))*2 + int(record.get('has_smartphone',False))*2
    )
    for col in ['education_level','social_category']:
        le = le_dict.get(col)
        try:    record[f'{col}_enc'] = int(le.transform([record.get(col,'')])[0])
        except: record[f'{col}_enc'] = 0

    feats = models['features_cluster']
    raw   = np.array([record.get(f,0) for f in feats]).reshape(1,-1)
    return models['scaler_cluster'].transform(raw)

CLUSTER_NAMES = {
    0: "High-Vulnerability Rural Poor",
    1: "Low-Income Rural Workers",
    2: "Low-Income Rural Workers",
    3: "High-Vulnerability Rural Poor",
    4: "High-Vulnerability Rural Poor",
}

X_clust = prepare_cluster_input(sample_applicant.copy())
cluster_id   = models['kmeans'].predict(X_clust)[0]
cluster_name = CLUSTER_NAMES.get(cluster_id, f"Cluster {cluster_id}")
pca_coords   = models['pca'].transform(X_clust)[0]

print("BENEFICIARY SEGMENTATION")
print(f"  Cluster ID:    {cluster_id}")
print(f"  Segment:       {cluster_name}")
print(f"  PCA coords:    PC1={pca_coords[0]:.3f}, PC2={pca_coords[1]:.3f}")

## 🚨 6. Anomaly / Fraud Detection (Isolation Forest)

In [ ]:
def prepare_anomaly_input(record: dict) -> np.ndarray:
    record['total_land_acres'] = record.get('wet_land_acres',0) + record.get('dry_land_acres',0)
    record['ineligibility_count'] = sum([
        record.get('has_four_wheeler',False), record.get('has_govt_employee_family',False),
        record.get('has_income_tax_filer',False), record.get('has_gst_business',False),
        record.get('already_receiving_pension',False),
    ])
    feats = models['features_anomaly']
    return np.array([record.get(f,0) for f in feats]).reshape(1,-1)

iso = models['isolation_forest']

# Normal applicant
normal_rec = sample_applicant.copy()
normal_rec['months_benefit_received'] = 10
X_norm = prepare_anomaly_input(normal_rec)
score_norm  = iso.decision_function(X_norm)[0]
flag_norm   = iso.predict(X_norm)[0]

# Suspicious applicant (high income + approved many months)
suspicious_rec = {**sample_applicant, 'annual_income_inr': 480000,
                   'electricity_units_year': 5500, 'months_benefit_received': 23,
                   'ineligibility_count': 2}
X_sus = prepare_anomaly_input(suspicious_rec)
score_sus = iso.decision_function(X_sus)[0]
flag_sus  = iso.predict(X_sus)[0]

print("ANOMALY DETECTION")
print(f"{'Record':<15} {'Score':>10}  {'Flag':>10}  {'Risk'}")
print("-"*55)
print(f"{'Normal':<15} {score_norm:>10.4f}  {flag_norm:>10}  {'✅ Normal' if flag_norm==1 else '🚨 Anomaly'}")
print(f"{'Suspicious':<15} {score_sus:>10.4f}  {flag_sus:>10}  {'✅ Normal' if flag_sus==1 else '🚨 Anomaly'}")
print()
print("Note: score < 0 → anomaly; more negative = more anomalous")

## 💬 7. GenAI RAG — Policy Q&A

In [ ]:
tfidf  = models['tfidf_vectorizer']
kb     = models['knowledge_base']
kb_mat = tfidf.transform(kb)

def rag_query(question: str, top_k: int = 2) -> dict:
    """
    Retrieve most relevant policy documents for a question.
    Returns context + similarity score.
    """
    q_vec = tfidf.transform([question])
    sims  = cosine_similarity(q_vec, kb_mat).flatten()
    top_i = sims.argsort()[-top_k:][::-1]
    return {
        'question':  question,
        'score':     float(sims[top_i[0]]),
        'context':   ' | '.join([kb[i] for i in top_i]),
        'top_docs':  [{'doc_id': int(i), 'score': float(sims[i]), 'text': kb[i][:120]} for i in top_i]
    }

questions = [
    "Who is eligible for the KMUT scheme?",
    "What is the monthly benefit amount?",
    "Can a widow apply?",
    "Which districts have high approval rates?",
    "What happens if I own a four-wheeler?",
    "How many beneficiaries are approved?",
]

print("RAG POLICY Q&A")
print("=" * 70)
for q in questions:
    r = rag_query(q)
    print(f"\nQ: {q}")
    print(f"   Score: {r['score']:.3f}")
    print(f"   A: {r['context'][:220]}...")

## 🔁 8. Batch Prediction on a DataFrame

In [ ]:
# Load dataset and run batch predictions
df = pd.read_csv('kmut_dataset.csv')

# Prepare features (already encoded in kmut_ml_dataset.csv)
# Quick demo using saved ML dataset
df_ml = pd.read_csv('kmut_ml_dataset.csv')

FEATURES_ELIG = models['features_eligibility']

X_batch = df_ml[FEATURES_ELIG].fillna(0).values
rf = models['rf_eligibility']

predictions  = rf.predict(X_batch)
probabilities = rf.predict_proba(X_batch)[:,1]

df_ml['predicted_eligible']    = predictions
df_ml['eligibility_confidence'] = probabilities

print(f"Batch prediction on {len(df_ml):,} records complete!")
print(f"\nPredicted eligible:     {predictions.sum():,} ({predictions.mean()*100:.1f}%)")
print(f"High confidence (>90%): {(probabilities>0.9).sum():,}")
print(f"Low confidence (<60%):  {(probabilities<0.6).sum():,}")
print()
print(df_ml[['applicant_id','district','annual_income_inr','final_eligible',
             'predicted_eligible','eligibility_confidence']].head(10).to_string(index=False))

## 📊 9. Full Pipeline — Single Applicant End-to-End

In [ ]:
def full_pipeline(record: dict) -> dict:
    """
    Run a single applicant through ALL models and return a complete profile.
    """
    results = {}

    # 1. Eligibility
    X_e = prepare_eligibility_input(record.copy())
    results['eligible_rf']     = bool(models['rf_eligibility'].predict(X_e)[0])
    results['eligible_prob']   = float(models['rf_eligibility'].predict_proba(X_e)[0][1])

    # 2. Approval (if applied)
    X_a = prepare_approval_input(record.copy())
    results['approval_gb']     = bool(models['gb_approval'].predict(X_a)[0])
    results['approval_prob']   = float(models['gb_approval'].predict_proba(X_a)[0][1])

    # 3. Cluster
    X_c = prepare_cluster_input(record.copy())
    cid = int(models['kmeans'].predict(X_c)[0])
    results['cluster_id']      = cid
    results['cluster_name']    = CLUSTER_NAMES.get(cid, f"Cluster {cid}")

    # 4. Anomaly
    r2 = record.copy()
    r2['months_benefit_received'] = record.get('months_benefit_received', 0)
    X_an = prepare_anomaly_input(r2)
    results['anomaly_flag']    = int(models['isolation_forest'].predict(X_an)[0])
    results['anomaly_score']   = float(models['isolation_forest'].decision_function(X_an)[0])
    results['is_anomaly']      = results['anomaly_flag'] == -1

    return results


# Test with sample applicant
applicant = {**sample_applicant, 'months_benefit_received': 8}
output = full_pipeline(applicant)

print("=" * 55)
print("FULL PIPELINE RESULTS — SINGLE APPLICANT")
print("=" * 55)
print(f"1. Eligibility:   {'✅ ELIGIBLE' if output['eligible_rf'] else '❌ NOT ELIGIBLE'}  ({output['eligible_prob']*100:.1f}% confidence)")
print(f"2. Approval:      {'✅ APPROVED' if output['approval_gb'] else '❌ NOT APPROVED'}  ({output['approval_prob']*100:.1f}% probability)")
print(f"3. Segment:       Cluster {output['cluster_id']} — {output['cluster_name']}")
print(f"4. Anomaly:       {'🚨 FLAGGED' if output['is_anomaly'] else '✅ NORMAL'}  (score={output['anomaly_score']:.3f})")

---
## 📌 Quick Reference Card

```python
import pickle
with open('kmut_models.pkl', 'rb') as f:
    models = pickle.load(f)

# Eligibility prediction
models['rf_eligibility'].predict(X)          # Random Forest
models['gb_eligibility'].predict(X)          # Gradient Boosting
models['dt_eligibility'].predict(X)          # Decision Tree
models['lr_eligibility'].predict(X_scaled)   # Logistic Regression (needs scaler)
models['scaler_eligibility'].transform(X)    # StandardScaler for LR

# Approval prediction
models['gb_approval'].predict(X)             # Gradient Boosting
models['rf_approval'].predict(X)             # Random Forest

# Clustering
models['scaler_cluster'].transform(X)        # MinMaxScaler
models['kmeans'].predict(X_scaled)           # K-Means cluster ID
models['pca'].transform(X_scaled)            # 2D coordinates

# Anomaly detection
models['isolation_forest'].predict(X)        # 1=normal, -1=anomaly
models['isolation_forest'].decision_function(X)  # anomaly score

# RAG
from sklearn.metrics.pairwise import cosine_similarity
q_vec = models['tfidf_vectorizer'].transform(["your question"])
sims  = cosine_similarity(q_vec, models['tfidf_vectorizer'].transform(models['knowledge_base']))

# Label encoders
models['label_encoders']['district'].transform(['Chennai'])

# Feature lists
models['features_eligibility']   # 21 features
models['features_approval']      # 22 features
models['features_cluster']       # 10 features
models['features_anomaly']       # 6 features
```